# AFD — Examples

A quick tour of the `afd` package (**A**tomistic **F**réchet **D**istance): FID-style
scoring for atomistic generative models, with a pretrained Conditional Equivariant
Transformer (CT) as the feature extractor instead of Inception.

1. Load a pretrained CT from HuggingFace.
2. Forward pass on a QM9 batch (non-periodic).
3. Forward pass on a tiny periodic crystal (verifies the PBC graph path).
4. Score a set of structures with `AFDScore`, then reproduce it with the low-level building blocks.

Run this notebook from the repository root so that `afd` is importable (the cell below
adds the root to `sys.path` as a fallback). Requirements: `torch`, `torch_geometric`,
`huggingface_hub`, `tqdm`.

## 0 · Setup

In [2]:
import os
import sys
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning)

# The notebook lives at the repo root; make `afd` importable even if the kernel
# was launched from elsewhere.
REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


## 1 · Load `ct-scd-pcq` from HuggingFace

Checkpoints live under `Ty-Perez/<model_name>` on the HuggingFace Hub:

| model name | trained on | use for |
|---|---|---|
| `ct-scd-pcq` | PCQM4Mv2 | small organic molecules (QM9-like) |
| `ct-scd-geom10` | GEOM (10 conformers / molecule) | drug-like molecules |
| `ct-scd-amp20` | Alexandria + MP-20 | periodic crystals |

We cache into the standard HuggingFace hub cache so the same download is reused by
`AFDScore` / `FeatureExtractor` later on.

In [4]:
from huggingface_hub.constants import HF_HUB_CACHE

from afd.models import load_pretrained_ct

model = load_pretrained_ct('ct-scd-pcq', cache_dir=HF_HUB_CACHE, device=DEVICE)
model.eval()

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Loaded CET with {num_params:.2f} M parameters')
print('rep_model:', type(model.rep_model).__name__)

Loaded CET with 10.06 M parameters
rep_model: ConditionalET


## 2 · Forward pass on a QM9 batch (non-periodic)

We use `torch_geometric.datasets.QM9` directly. `AddStandardKeys` populates the
`pbc / natoms / cell` attributes that the CT's `GraphGenerator` requires for
non-periodic molecules (a zero `pbc` flag and a padded bounding-box `cell`).

The first run downloads and processes QM9 into `./tmp/qm9` (a few minutes).

In [6]:
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader

from afd.models import AddStandardKeys

dataset = QM9(root='./tmp/qm9', transform=AddStandardKeys())
print('QM9 size:', len(dataset))

loader = DataLoader(dataset, batch_size=16, shuffle=False)
batch = next(iter(loader)).to(DEVICE)
print('batch:', batch)

with torch.no_grad():
    out = model(
        z=batch.z,
        pos=batch.pos,
        batch=batch.batch,
        graph_batch=batch,
        return_atom_embs=True,
    )

for k, v in out.items():
    print(f'  {k:12s}: {tuple(v.shape) if v is not None else None}')

QM9 size: 130831
batch: DataBatch(x=[101, 11], edge_attr=[172, 4], y=[16, 19], pos=[101, 3], z=[101], smiles=[16], name=[16], idx=[16], pbc=[16, 3], natoms=[16], cell=[16, 3, 3], batch=[101], ptr=[17])
  noise_pred  : (101, 3)
  mol_emb     : (16, 256)
  atom_embs   : (101, 256)
  y           : (16, 1)


`mol_emb` is the 256-d graph-level embedding that AFD is computed on. `atom_embs` are
the per-atom features it is pooled from, and `noise_pred` / `y` are the denoising
heads the CT was pretrained with (not used by AFD).

## 3 · Forward pass on a periodic crystal

Sanity-check that the periodic graph path also works. We build a toy 4-atom cubic
cell with PBC on all axes and pass it through the same model. (For real materials
scoring you would use the `ct-scd-amp20` checkpoint; see §4.)

In [9]:
from torch_geometric.data import Batch, Data

pos = torch.tensor([[0.0, 0.0, 0.0], [2.5, 2.5, 0.0], [2.5, 0.0, 2.5], [0.0, 2.5, 2.5]])
z   = torch.tensor([6, 6, 6, 6])
cell = torch.eye(3).unsqueeze(0) * 5.0
pbc  = torch.tensor([[True, True, True]])
natoms = torch.tensor([4], dtype=torch.long)

data = Data(z=z, pos=pos, cell=cell, pbc=pbc, natoms=natoms)
batch = Batch.from_data_list([data]).to(DEVICE)

with torch.no_grad():
    out = model(z=batch.z, pos=batch.pos, batch=batch.batch, graph_batch=batch)

print('mol_emb (periodic):', tuple(out['mol_emb'].shape))
print('noise_pred       :', tuple(out['noise_pred'].shape))

mol_emb (periodic): (1, 256)
noise_pred       : (4, 3)


## 4 · Scoring with `AFDScore`

`AFDScore` bundles everything: it downloads precomputed **reference** features for a
canonical dataset (`"qm9"`, `"mp20"` or `"geom"`, from the `Ty-Perez/AtomisticEval`
dataset repo), picks the matching CT checkpoint, and exposes a callable that
featurizes any candidate set and returns the Fréchet distance to the reference.

A candidate set is anything indexable that yields `torch_geometric.data.Data`
objects or dicts with at least `z` and `pos` (plus `cell` / `pbc` for periodic
structures). Here we use a slice of QM9 itself as a stand-in for generated samples. Those
molecules are also part of the reference set, so the score is a pure
finite-sample floor: the value a perfect generator would reach at this N.

> **AFD is biased in the sample size N.** Never compare scores computed at different
> N. Pass `ref_sample_size` to subsample the reference to the candidate size (the
> scorer also trims both sides to the smaller N by default), and report N with
> every number.

In [11]:
from afd import AFDScore

scorer = AFDScore('qm9', device=DEVICE, batch_size=256)
print('reference features:', tuple(scorer.ref_features.shape))

N = 5_000
plain_qm9 = QM9(root='./tmp/qm9')          # no transform: AFDScore applies AddStandardKeys itself
candidate = plain_qm9[-N:]                 # last 5k molecules as a fake "generation" set

score = scorer(candidate, ref_sample_size=N, seed=0)
print(f'AFD@{N} (QM9 slice vs QM9 reference): {score:.4f}')

reference features: (130831, 256)
AFD@5000 (QM9 slice vs QM9 reference): 0.0415


### The same number from the low-level pieces

`AFDScore` is a thin wrapper around three functions you can also call directly:

- `compute_features(records, model, transform=AddStandardKeys())` runs the CT and stacks `mol_emb`.
- `gaussian_stats(feats)` returns the mean and covariance.
- `frechet_distance(...)` / `afd_score(feats_ref, feats_gen)` evaluate the closed-form distance.

`FeatureExtractor(data_keyword='qm9')` is a convenience that loads the checkpoint and
applies the transform for you; below we reuse the `model` from §1 instead.

In [13]:
from afd import compute_features, afd_score, gaussian_stats, frechet_distance

ref_feats = scorer.sample_ref(N, seed=0)
gen_feats = compute_features(candidate, model, transform=AddStandardKeys(),
                             batch_size=256, device=DEVICE, desc='candidate')
print('ref  features:', tuple(ref_feats.shape))
print('gen  features:', tuple(gen_feats.shape))

print(f'afd_score          : {afd_score(ref_feats, gen_feats):.4f}')

mu_r, cov_r = gaussian_stats(ref_feats)
mu_g, cov_g = gaussian_stats(gen_feats)
print(f'frechet_distance   : {frechet_distance(mu_r, cov_r, mu_g, cov_g):.4f}')

ref  features: (5000, 256)
gen  features: (5000, 256)
afd_score          : 0.0415
frechet_distance   : 0.0415


### Sample-size bias, illustrated

Scoring the *same* candidate set at different N moves the number substantially,
which is why N must be fixed across every comparison.

In [15]:
for n in (500, 1_000, 2_500, 5_000):
    s = afd_score(ref_feats[:n], gen_feats[:n])
    print(f'N = {n:>5d}   AFD = {s:.4f}')

N =   500   AFD = 0.0642
N =  1000   AFD = 0.0531
N =  2500   AFD = 0.0431
N =  5000   AFD = 0.0415


## 5 · Scoring your own generations

**In Python**, pass any list of records:

```python
records = torch.load('my_gens.pt', weights_only=False)   # list of {'z': ..., 'pos': ...[, 'cell', 'pbc']}
scorer = AFDScore('qm9', device='cuda:0')
print(scorer(records, ref_sample_size=len(records), seed=0))
```

Use `AFDScore('mp20')` for periodic crystals (checkpoint `ct-scd-amp20`), or pass your
own reference dataset together with `data_modality='molecules'` / `'materials'`:

```python
scorer = AFDScore(my_reference_records, data_modality='materials')
```

**From the command line**, the same scorer is exposed as `python -m afd`:

```bash
python -m afd score --ref qm9 --gen my_gens.pt other_gens.pt \
    --ref-sample-size 5000 --seed 0 --device cuda:0
```

Run `python -m afd score --help` for all options.